<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/RMS-Norm/rms-norm.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("rms-norm", RMSNorm)`

# Implement RMS Norm

**Difficulty**: 🟢 Easy

---

### Problem Statement

**RMS Norm** (Root Mean Square Layer Normalization) is a simplification of LayerNorm used in modern transformer architectures like LLaMA, Mistral, and Gemma. Instead of computing both mean and variance, it normalizes by the root mean square of the input, making it faster and simpler while preserving performance.

Your task is to implement an `RMSNorm` module from scratch as a `torch.nn.Module`.

---

### Requirements

1. **RMS Computation** — Compute the root mean square: `RMS(x) = sqrt(mean(x^2) + eps)` along the last dimension.
2. **Normalization** — Divide the input by its RMS value.
3. **Learnable Scale** — Apply a learnable per-feature scale parameter (gamma), initialized to ones.
4. **Shape Preservation** — Output shape must match input shape.

---

### Constraints

- ✅ Use `torch.nn.Parameter` for the learnable scale.
- ✅ Must handle any input shape `(..., dim)` — the last dimension is normalized.
- ✅ Use `eps` (default `1e-8`) for numerical stability.
- ❌ Do **not** use `torch.nn.RMSNorm` or `torch.nn.LayerNorm`.

---

### Formula

```
RMSNorm(x) = (x / RMS(x)) * gamma

where RMS(x) = sqrt(mean(x^2, dim=-1) + eps)
```

---

<details>
  <summary>💡 Hint</summary>

  - Square the input, take the mean along the last dimension, add epsilon, then take the square root.
  - Use `keepdim=True` in the mean so broadcasting works when you divide.
  - The scale parameter `gamma` should have shape `(dim,)` and be created with `nn.Parameter(torch.ones(dim))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        # TODO: Create a learnable scale parameter (gamma) of shape (dim,), initialized to ones
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: shape (..., dim)
        # TODO: Step 1 - Compute the RMS: sqrt(mean(x^2) + eps) along the last dimension
        ...
        # TODO: Step 2 - Normalize x by dividing by the RMS, then multiply by gamma
        ...

In [ ]:
# === Tests ===
import torch

# Test 1: Output shape
x = torch.randn(3, 5)
rmsnorm = RMSNorm(dim=5)
out = rmsnorm(x)
assert out.shape == x.shape, f"Output shape {out.shape} != input shape {x.shape}"
print("PASSED: output shape matches input shape")

# Test 2: Learnable parameter exists and has correct shape
assert hasattr(rmsnorm, 'scale') or hasattr(rmsnorm, 'weight') or hasattr(rmsnorm, 'gamma'), \
    "RMSNorm should have a learnable scale parameter"
params = list(rmsnorm.parameters())
assert len(params) == 1, f"Expected 1 learnable parameter, got {len(params)}"
assert params[0].shape == (5,), f"Scale parameter shape should be (5,), got {params[0].shape}"
print("PASSED: learnable scale parameter has correct shape")

# Test 3: Higher-dimensional input
x_3d = torch.randn(2, 4, 8)
rmsnorm_3d = RMSNorm(dim=8)
out_3d = rmsnorm_3d(x_3d)
assert out_3d.shape == (2, 4, 8), f"3D output shape mismatch: {out_3d.shape}"
print("PASSED: handles 3D input correctly")

# Test 4: Not all zeros, not NaN
assert not torch.isnan(out).any(), "Output contains NaN"
assert not torch.isinf(out).any(), "Output contains Inf"
assert out.abs().sum() > 0, "Output is all zeros"
print("PASSED: output is valid (no NaN, Inf, or all-zeros)")

# Test 5: Scale parameter is learnable (gradient flows)
x = torch.randn(4, 8, requires_grad=True)
rmsnorm8 = RMSNorm(dim=8)
loss = rmsnorm8(x).sum()
loss.backward()
assert list(rmsnorm8.parameters())[0].grad is not None, "Gradient did not flow to scale parameter"
print("PASSED: gradient flows through scale parameter")

print("\nAll tests passed!")